In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import imageio.v2 as imageio
import os

# ======================================================
# 1️⃣ CONFIGURACIÓN DE DATOS (Entremezclados)
# ======================================================
torch.manual_seed(0)
X, y = make_moons(n_samples=600, noise=0.30, random_state=30)
X_tensor = torch.tensor(X.astype(np.float32))
y_tensor = torch.tensor(y.astype(np.float32).reshape(-1, 1))

# ======================================================
# 2️⃣ DEFINICIÓN DE CAPAS Y MODELOS
# ======================================================

# --- MLP (Multi-Layer Perceptron) ---
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(64,)):  # puedes poner (64, 64) si quieres 2-64-64-1
        super().__init__()
        layers = []
        in_dim = 2
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.Tanh()]
            in_dim = h
        layers += [nn.Linear(in_dim, 1)]
        self.net = nn.Sequential(*layers)

        # string de arquitectura tipo "2-64-1" o "2-64-64-1"
        self.arch_str = "2-" + "-".join(map(str, hidden_sizes)) + "-1"

    def forward(self, x):
        return self.net(x)

# --- KAN (Kolmogorov-Arnold Network) con Splines ---
class SplineLayer(nn.Module):
    def __init__(self, in_features, out_features, grid_size=10):
        super().__init__()
        self.knots = nn.Parameter(torch.linspace(-3, 3, grid_size))
        self.coeffs = nn.Parameter(torch.randn(in_features, grid_size, out_features) * 0.1)

    def forward(self, x):
        basis = torch.exp(-(x.unsqueeze(-1) - self.knots)**2)
        return torch.einsum('bij,ijo->bo', basis, self.coeffs)

class KAN(nn.Module):
    def __init__(self, hidden_sizes=(64,)):  # análogo: (64, 64) -> 2-64-64-1
        super().__init__()
        # construimos capas spline según hidden_sizes
        dims = [2] + list(hidden_sizes) + [1]
        self.layers = nn.ModuleList([SplineLayer(dims[i], dims[i+1]) for i in range(len(dims)-1)])

        self.arch_str = "-".join(map(str, dims))  # "2-64-1" o "2-64-64-1"

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = torch.tanh(x)
        return x

# ======================================================
# 3️⃣ CREACIÓN DE LOS OBJETOS (Antes de entrenar)
# ======================================================
mlp_model = MLP(hidden_sizes=(32,))      # cambia a (64, 64) si quieres 2-64-64-1
kan_model = KAN(hidden_sizes=(32,))      # cambia a (64, 64) si quieres 2-64-64-1

# ======================================================
# 4️⃣ FUNCIÓN PARA ENTRENAR Y CREAR EL GIF
# ======================================================
def train_and_animate(model, name="Model", arch="", epochs=300):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCEWithLogitsLoss()
    frames = []

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()].astype(np.float32))

    print(f"Generando GIF para {name} (Entrenamiento)...")
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            fig, ax = plt.subplots(figsize=(6, 5))
            with torch.no_grad():
                Z = (torch.sigmoid(model(grid)) > 0.5).float().reshape(xx.shape).numpy()

            ax.contourf(xx, yy, Z, alpha=0.1, cmap='RdBu')
            ax.contour(xx, yy, Z, colors='red', linewidths=1.5)

            y_p = y.squeeze()
            ax.scatter(X[y_p==0, 0], X[y_p==0, 1], c='blue', edgecolors='k', s=25, label='Clase A')
            ax.scatter(X[y_p==1, 0], X[y_p==1, 1], c='pink', edgecolors='k', s=25, label='Clase B')

            # ✅ Título con arquitectura
            ax.set_title(f"{name} ({arch}) | Época {epoch} | Loss: {loss.item():.4f}")
            ax.legend(loc='upper right')

            fig.canvas.draw()
            image = np.frombuffer(fig.canvas.buffer_rgba(), dtype='uint8')
            image = image.reshape(fig.canvas.get_width_height()[::-1] + (4,))
            frames.append(image[:, :, :3])
            plt.close(fig)

    imageio.mimsave(f'evolucion_{name}.gif', frames, fps=12)
    print(f"✅ ¡Archivo creado con éxito: evolucion_{name}.gif!")

# ======================================================
# 5️⃣ EJECUCIÓN
# ======================================================
train_and_animate(mlp_model, name="MLP", arch=mlp_model.arch_str, epochs=600)
train_and_animate(kan_model, name="KAN", arch=kan_model.arch_str, epochs=600)

Generando GIF para MLP (Entrenamiento)...
✅ ¡Archivo creado con éxito: evolucion_MLP.gif!
Generando GIF para KAN (Entrenamiento)...
✅ ¡Archivo creado con éxito: evolucion_KAN.gif!
